# COMP5339 Assignment 2

GROUP: TUT11-ASSIGNMENTGRP-10 <br>

SID <br>
- 540969766
- 540931475

In [17]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import ast
import paho.mqtt.client as mqtt
import json


### Class

In [ ]:
class Dataloader:
    '''
    Handle data loading and preprocessing
    '''
    def __init__(self):
        '''
        Initialize the Dataloader with the API key  
        '''
        load_dotenv()
        api_key = os.getenv("API_KEY")

        if not api_key:
            raise ValueError("API_KEY is not set in the environment variable. Check .env file.")
        
        self.api_key = api_key
        self.base_url = "https://api.openelectricity.org.au/v4"
        self.headers = {'Authorization': f'Bearer {api_key}'}
        self.delay = 60

        print("Ready to load data")
    

    def save_to_csv(self, df, filename):
        '''
        Save the dataframe to a csv file
        '''
        if not df.empty:
            df.to_csv(filename, index = False)
            print(f'Successfully saved data to {filename}')
        else:
            print("The dataframe is empty. No data to save.")



    def get_facilities(self):
        ''' 
        Get the facilities data from the API
        '''
        facilities_url = f'{self.base_url}/facilities/'
        params = {
            'network_id': 'NEM'
        }

        try:
            response = requests.get(facilities_url, headers = self.headers, params = params)
            facilities = pd.DataFrame(response.json())

            print(f'Successfully retrieved data for {len(facilities)} facilities')

            self.save_to_csv(facilities, 'facilities_raw.csv')
            
            time.sleep(self.delay) # wait for 60 seconds before the next request

            return facilities
        
        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None
        


    def cleaning_facilities(self, facilities):
        '''
        Clean the facilities data
        '''
        cleaned_facilities = []

        for idx, row in facilities.iterrows():

            try:
                if isinstance(row['data'], dict):
                    facility_data = row['data']
                elif isinstance(row['data'], str):
                    facility_data = ast.literal_eval(row['data'])
                else:
                    print(f"⚠ Unexpected type at row {idx}")
                    continue    

                # Extract the basic information first
                facility_code = facility_data.get('code')
                facility_name = facility_data.get('name')
                facility_region = facility_data.get('network_region')

                # Extract the location
                location = facility_data.get('location', {})
                latitude = location.get('lat')
                longitude = location.get('lng')

                # Get the unit details
                units = facility_data.get('units', [])


                for unit in units:
                    unit_info = {
                        # Facility Unit information
                        'facility_code': facility_code,
                        'facility_name': facility_name,
                        'facility_region': facility_region,
                        'latitude': latitude,
                        'longitude': longitude,

                        # Facility power, emissions
                        'facility_power': 0,
                        'facility_emissions': 0,
                        
                        # Unit information
                        'unit_code': unit.get('code'),
                        'unit_status': unit.get('status_id'),
                        'fuel_type': unit.get('fueltech_id'),
                        'capacity_registered': unit.get('capacity_registered'),
                        'capacity_maximum': unit.get('capacity_maximum'),
                        'emissions_co2': unit.get('emissions_factor_co2'),

                        # Unit power, emissions
                        'unit_power': 0,
                        'unit_emissions': 0                      
                    }

                    cleaned_facilities.append(unit_info)
            
            except Exception as e:
                print(f"Error processing facility {idx}: {e}")
                continue
        
        self.save_to_csv(pd.DataFrame(cleaned_facilities), 'facilities_cleaned.csv')
        

        return pd.DataFrame(cleaned_facilities)
            


    def get_facility_details(self, facility_code, interval, start_date, end_date):
        '''
        Get the facility power generation data
        '''
        facility_url = f'{self.base_url}/data/facilities/NEM'
        params = {
            'metrics': ['power', 'emissions'],
            'interval': interval,
            'date_start': start_date,
            'date_end': end_date,
            'facility_code': facility_code
        }

        try:
            response = requests.get(facility_url, headers = self.headers, params = params)

            time.sleep(self.delay) # wait for 60 seconds before the next request

            facility_data = response.json()


            records = {}

            if 'data' in facility_data:
                for metrics in facility_data['data']:
                    metric_name = metrics.get('metric') # Get power or emissions

                    for record in metrics.get('results', []):
                        unit_code = record['columns']['unit_code']

                        for timestamp, value in record['data']:
                            key = (timestamp, unit_code)

                            if key not in records:
                                records[key] = {
                                    'interval': timestamp,
                                    'unit_code': unit_code,
                                    'unit_power': 0,
                                    'unit_emissions': 0
                                }
                            
                            if metric_name == 'power':
                                records[key]['unit_power'] = value
                            elif metric_name == 'emissions':
                                records[key]['unit_emissions'] = value

            
            unit_data = pd.DataFrame(list(records.values()))

            if not unit_data.empty:
                # Preprocess -> clean negative emissions
                neg_emissions = (unit_data['unit_emissions'] < 0).sum()
                if neg_emissions > 0:
                    unit_data.loc[unit_data['unit_emissions'] < 0, 'unit_emissions'] = 0
                    
                print(f'Successfully retrieved {len(unit_data)} records for facility {unit_data['unit_code'].nunique()} units from {len(facility_code)} facilities!')

            return unit_data
        
        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None
        


    def get_market_data(self, interval, start_date, end_date):
        '''
        Get the market price and demand data
        '''
        market_url = f'{self.base_url}/market/network/NEM'
        params = {
            'metrics': ['price', 'demand'],
            'interval': interval,
            'date_start': start_date,
            'date_end': end_date,
            'primary_grouping': 'network_region'
        }

        try:
            response = requests.get(market_url, headers = self.headers, params = params)
            market_data = response.json()

            time.sleep(self.delay) # wait for 60 seconds before the next request
            
            records = {}

            if 'data' in market_data:
                for metrics in market_data['data']:
                    metric_name = metrics.get('metric') # Get price or demand

                    for record in metrics.get('results', []):
                        region = record['columns']['region']

                        for timestamp, value in record['data']:
                            key = (timestamp, region)

                            if key not in records:
                                records[key] = {
                                    'interval': timestamp,
                                    'region': region,
                                    'market_price': 0,
                                    'market_demand': 0
                                }
                            
                            if metric_name == 'price':
                                records[key]['market_price'] = value
                            elif metric_name == 'demand':
                                records[key]['market_demand'] = value
            
            market = pd.DataFrame(list(records.values()))

            if not market.empty:
                print(f'Successfully retrieved {len(market)} records for {len(market['region'].unique())} regions!')
            
            return market

        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None


    def merge_data(self, facilities, units, market):
        '''
        Merge the facilities, units, and market data
        '''
        
        # Find the facilities that has working units
        units_working = set(units['unit_code'].unique())
        all_units = set(facilities['unit_code'].unique())
        units_notworking = all_units - units_working

        print(f"Number of working units: {len(units_working)}")
        print(f"Number of non-working units: {len(units_notworking)}")

        # Merge the facilities and units data (those with working units)
        merged = pd.merge(units, facilities[['facility_code', 'facility_name', 'facility_region', 
                                            'latitude', 'longitude', 'unit_code', 'fuel_type', 'unit_status']], on = 'unit_code', how = 'left')

        # Check merge quality
        missing_facility = merged['facility_code'].isna().sum()
        print(f"   Total records: {len(merged)}")
        print(f"   Missing facility_code: {missing_facility}")
        
        if missing_facility > 0:
            print(f"\nWARNING: {missing_facility} records missing facility details!")
            print("\nSample:\n")
            print(merged[merged['facility_code'].isna()][['unit_code', 'unit_power', 'unit_emissions']].head())

        # Add interval to the not-working units data
        if units_notworking:

            # Set the interval
            intervals = units['interval'].unique()
            
            # Extract the not-working units
            units_notworking = facilities[facilities['unit_code'].isin(units_notworking)][['facility_code', 'facility_name', 'facility_region', 
                                                                                            'latitude', 'longitude', 'unit_code', 'fuel_type']].drop_duplicates()

            # Cross-join the not-working units with the intervals
            expanded = []
            for interval in intervals:
                temp = units_notworking.copy()
                temp['interval'] = interval
                expanded.append(temp)

            units_notworking = pd.concat(expanded, ignore_index = True)

            # Merge with the merged data
            merged = pd.concat([merged, units_notworking], ignore_index = True)
        
        # Convert the NaN to 0
        merged['unit_power'] = merged['unit_power'].fillna(0)
        merged['unit_emissions'] = merged['unit_emissions'].fillna(0)

        # Calculate the total power and emissions for each facility
        per_facility = merged.groupby(['interval', 'facility_code']).agg({
            'unit_power': 'sum',
            'unit_emissions': 'sum'
        }).reset_index()

        per_facility.columns = ['interval', 'facility_code', 'facility_power', 'facility_emissions']

        # Merge the total power and emissions with merged unit data
        merged = pd.merge(merged, per_facility, on = ['interval', 'facility_code'], how = 'left')


        # Merge market data
        merged = pd.merge(merged, market, left_on = ['interval', 'facility_region'], right_on = ['interval', 'region'], how = 'left')
        # Drop the region column
        if 'region' in merged.columns:
            merged = merged.drop(columns = ['region'])

        # Convert the NaN to 0
        merged['market_price'] = merged['market_price'].fillna(0)
        merged['market_demand'] = merged['market_demand'].fillna(0)

        # Calculate the total power and emissions for each facility
        per_facility = merged.groupby(['interval', 'facility_code']).agg({
            'facility_power': 'sum',
            'facility_emissions': 'sum'
        }).reset_index()

        column_order = ['interval', 'facility_region', 'market_price', 'market_demand', 'facility_code', 'facility_name', 'latitude', 'longitude', 'facility_power', 'facility_emissions', 'unit_code', 'fuel_type', 'unit_power', 'unit_emissions']

        merged = merged[column_order]
        merged = merged.sort_values(by = ['facility_code', 'unit_code', 'interval'])

        return merged


class MQTT:

    def setup_mqtt(self, group_name = 'tut11_grp10'):
        '''
        Setup the MQTT client
        '''
        self.group_name = group_name
        self.mqtt_client = None
        self.mqtt_broker = 'test.mosquitto.org'
        self.mqtt_port = 1883
        self.mqtt_topic = f'comp5339/{group_name}/power_emissions'

        print(f'MQTT client setup for group {group_name} for topic {self.mqtt_topic}')


    def connect_mqtt(self):
        '''
        MQTT connection to the broker
        '''
        try:
            self.mqtt_client = mqtt.Client()
            self.mqtt_client.connect(self.mqtt_broker, self.mqtt_port, 60)
            self.mqtt_client.loop_start()

            print(f'Connected to MQTT broker {self.mqtt_broker} on port {self.mqtt_port}')
            return True

        except Exception as e:
            print(f'Failed to connect to MQTT broker: {e}')
            return False
    
    def disconnect_mqtt(self):
        '''
        Disconnect from the MQTT broker
        '''
        if self.mqtt_client:
            self.mqtt_client.loop_stop()
            self.mqtt_client.disconnect()
            print('Disconnected from MQTT broker')
        else:
            print('No MQTT client to disconnect')
    

    def publish_message(self, data, delay = 0.1):
        '''
        Publish a message to the MQTT broker
        '''
        if data.empty or self.mqtt_client is None:
            print('No data to publish or MQTT client not connected')
            return 0
        
        print(f'Publishing {len(data)} messages to {self.mqtt_topic}')

        published = 0
        

        # Deal with the NaN and Inf values
        for idx, row in data.iterrows():
            message = row.to_dict()

            for key, value in message.items():
                if isinstance(value, (int, float)):
                    if np.isnan(value) or np.isinf(value):
                        message[key] = 0
                    else:
                        message[key] = float(value)
        
        try:
            payload = json.dumps(message, default = str)
            result = self.mqtt_client.publish(self.mqtt_topic, payload)

            if result.rc == mqtt.MQTT_ERR_SUCCESS:
                published += 1
                print(f'Published message {published} to {self.mqtt_topic}')
            else:
                print(f'Failed to publish message {idx} to {self.mqtt_topic}: {result.rc}')

            time.sleep(delay) # by default it's 0.1 second

        except Exception as e:
            print(f'Error publishing message {idx} to {self.mqtt_topic}: {e}')
        
        return published
    

class DataProcessor(Dataloader, MQTT):
    '''
    Integrate the data from the API and publish to the MQTT broker
    '''

    def __init__(self, group_name = 'tut11_grp10'):
        # Initialize the parent classes
        super().__init__()

        # Add MQTT client
        self.setup_mqtt(group_name)

## Data Retrieval

### Preprocessing for metadata
- negative price: oversupply
- negative power: consume more than generated

In [90]:
loader = DataProcessor(group_name = 'tut11_grp10')

# Get the facilities data
facilities = loader.get_facilities()

facilities_clean = loader.cleaning_facilities(facilities)

# # Publish the data to the MQTT broker
# loader.publish_message(full_data)
# loader.disconnect_mqtt()

Ready to load data
MQTT client setup for group tut11_grp10 for topic comp5339/tut11_grp10/power_emissions
Successfully retrieved data for 515 facilities
Successfully saved data to facilities_raw.csv
Successfully saved data to facilities_cleaned.csv


In [91]:
facilities_clean.head()

,facility_code,facility_name,facility_region,latitude,longitude,facility_power,facility_emissions,unit_code,unit_status,fuel_type,capacity_registered,capacity_maximum,emissions_co2,unit_power,unit_emissions
0,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV1,operating,solar_utility,24.75,19.00,NaN,0,0
1,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV2,operating,solar_utility,0.20,0.20,NaN,0,0
2,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPPV3,operating,solar_utility,0.02,0.02,NaN,0,0
3,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPBA1G,operating,battery_discharging,7.76,6.15,NaN,0,0
4,ADP,Adelaide Desalination,SA1,-35.096948,138.484061,0,0,ADPBA1L,operating,battery_charging,7.76,6.15,NaN,0,0


In [56]:
facilities_clean[facilities_clean.facility_code == 'ANGASTON'].unit_code

8      ANGAS1
9      ANGAS2
10    ANGAST1
Name: unit_code, dtype: object

In [87]:
# samples = facilities_clean[0:11]
facility_codes = facilities_clean['facility_code'].unique()

# Get the facility details
units = loader.get_facility_details(facility_codes, '1d', '2025-10-01T00:00:00', '2025-10-07T23:59:59')
units

# # Get the market data
# market = loader.get_market_data('1d', '2025-10-01T00:00:00', '2025-10-07T23:59:59')

# # Merge the data
# full_data = loader.merge_data(samples, units, market)
# market


""


In [88]:
units

# full_data2 = loader.merge_data(samples, units, market)
# full_data2

""


## Data Integration and Materialisation/Cashing

- whether the per-facility power generated is power or energy
  - power: Instantaneous power output/consumption (MW)
  - energy: Energy generated/consumed over time (MWh)

- per-market price and demand -> is this about price and demand per region (NSW, SA) or per fuel type (when i check the docs it's only filtered by region)

- Is it okay to keep only those facilities that has working (operating) as those have no working unit don't have any power, emission data
- What does it mean by continuous streaming

- for visualisation:
  - is it okay to show powerstation name and current power output or emissions or should we have to keep those information floating always
  - the latest power production and emissions data meaning the overall data for that certain 🌏

In [77]:
class OpenElectricity:
    def __init__(self, max_retries=3, retry_delay=1, start_date=None, end_date=None):
        # get whole facilities
        self.url_facilities = "https://api.openelectricity.org.au/v4/facilities/"
        # get facilities details based on metrics
        self.url_facilities_power = "https://api.openelectricity.org.au/v4/data/facilities/"
        # get price and demand url
        self.url_network = "https://api.openelectricity.org.au/v4/market/network/"

        # api key
        self.key = os.getenv("API_KEY")
        # set maximum retries if failed to retrieve the data
        self.max_retries = max_retries
        # set delay for each api hit
        self.retry_delay = retry_delay

        # static date for start_date
        self.date_start = start_date
        # static date for end_date
        self.date_end = end_date

    def to_csv(self, dataframe: pd.DataFrame, filename):
        dataframe.to_csv(filename, index=False)
        print(f"Saved to {filename}")

    def safe_request(self, url, headers, params):
        """Handles retries and exceptions for API calls."""
        for attempt in range(1, self.max_retries + 1):
            try:
                response = requests.get(url, headers=headers, params=params)
                if response.status_code in [200, 416]:
                    return response
                else:
                    print(f"Attempt {attempt}: API responded with {response.status_code}", response)
            except requests.exceptions.Timeout:
                print(f"Attempt {attempt}: Request timed out, retrying in {self.retry_delay}s...")
            except requests.exceptions.ConnectionError:
                print(f"Attempt {attempt}: Connection error, retrying in {self.retry_delay}s...")
            except Exception as e:
                print(f"Attempt {attempt}: Unexpected error: {e}")

            time.sleep(self.retry_delay)

        print("All retries failed. Skipping this request.")
        return None

    def get_facility(self):
        url = self.url_facilities
        headers = {"Authorization": f"Bearer {self.key}"}
        params = {
            'interval': '5m',
            'network_id': 'NEM'
        }
        
        response = self.safe_request(url, headers, params)
        if response:
            df = pd.DataFrame(response.json())
            time.sleep(self.retry_delay)
            return df
        return pd.DataFrame()

    def get_facility_data(self, facility_code:list):
        url = f"{self.url_facilities_power}NEM"
        headers = {"Authorization": f"Bearer {self.key}"}
        params = {
            'interval': '5m',
            'date_start': self.date_start,
            'date_end': self.date_end,
            'metrics': ['power', 'emissions'], 
            'facility_code': facility_code      
            }

        response = self.safe_request(url, headers, params)
        #response = requests.get(url, headers=headers, params=params)
        if response:
            df = pd.DataFrame(response.json())
            time.sleep(self.retry_delay)
            return df
        return pd.DataFrame()
    
    def get_network_data(self, network_region):
        url = f"{self.url_network}NEM"
        headers = {"Authorization": f"Bearer {self.key}"}
        params = {
            'interval': '5m',
            'date_start': self.date_start,
            'date_end': self.date_end,
            'metrics': ['price','demand'],
            'network_region':network_region
        }
        
        response = self.safe_request(url, headers, params)
        if response:
            df = pd.DataFrame(response.json())
            time.sleep(self.retry_delay)
            return df
        return pd.DataFrame()

In [84]:
facility_codes = facilities_clean['facility_code'].unique()
facility_codes

array(['ADP', 'ALDGASF', 'AMCORGR', 'ANGASTON', 'APS', 'APPIN', 'ARWF',
       'AVLSF', 'AWABAREF', 'DEIBDL', 'BAKING', 'BHWF', 'BALBESS',
       'BBASEHOS', 'BANGOWF', 'BAPS', 'BANKSPT', 'BANNSP', 'BARCALDN',
       'BARCSF', 'BARKIPS', 'BARRON', 'BASTYAN', 'BAYSW', 'BBDISEL1',
       'BELLBAY', 'BBP31', 'BRYB1WF1', 'BERWICK', 'BERYLSF', 'BIALAWF',
       'BLAYNEY', 'SNOWY6', 'BLUEGSF', 'BLULAKE', 'BLYTHB', 'BOCOROCK',
       'BODWF', 'MCKAY', 'BOLIVAR', 'BOLIVPS', 'BOMENSF', '0BCWF',
       'BBATTERY', 'BRAEMARA', 'B2PS', 'BRNDBES', 'BROADMDW', '0BSSF',
       'BWTR1', 'BHILLGT', 'BROKENH', 'BHB', 'BROOKLYN', 'BROWNMT',
       'BPLANDF', '0WELLINGTONBESS', 'BULGANA', '0BUNDSF', 'BNGSF1',
       'BNGSF2', '0BUNGAMA', 'BDONGHYD', 'BURRIN', 'BUTLERSG',
       '0CALBESS1', '0CALBESS2', 'CALL_A', 'CALL_B', 'CALLIDEC1',
       'CANUNDA1', 'CAPBES', 'CAPTL_WF', 'CESF', 'LI_WY_CA', 'CATHROCK',
       'CTHLWF', 'CETHANA', 'CHALLWF', 'CHPSTWF', 'CHYTWF', 'CHILDSF',
       'CHBESS', 'CBWWBA', '

In [85]:
start_date ="2025-10-01T00:00:00"
end_date = "2025-10-08T00:00:00"

api = OpenElectricity(start_date=start_date, end_date=end_date)

# get all facility data
df = api.get_facility()
x_df = api.get_facility_data(facility_codes)

Attempt 1: API responded with 400 <Response [400]>
Attempt 2: API responded with 400 <Response [400]>
Attempt 3: API responded with 400 <Response [400]>
All retries failed. Skipping this request.


In [83]:
for i in x_df['data'][0]['results']:
    print(i['name'])

power_ADPBA1
power_ADPBA1G
power_ADPPV1
power_ALDGASF1
power_ANGAST1
power_ADPBA1L
